In [ ]:
import subprocess
import math
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())


In [ ]:
audio = r"video.mp4"
output_srt = audio.replace(".mp4", "_transcribed.srt")
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from faster_whisper import WhisperModel

In [ ]:
# --- Load model ---
# Available models: tiny, base, small, medium, large-v3, large-v3-turbo, distil-large-v3
model = WhisperModel("medium", device="cuda", compute_type="int8_float16")

# --- Run translation ---
# val_filter = True can Optimized for GPU speed & low VRAM usage
segments, info = model.transcribe(audio, task="transcribe", beam_size=5, vad_filter=True, initial_prompt='customize here')

print("Detected language:", info.language)

# the probability/confidence of detection
print("Language probability:", info.language_probability)

In [ ]:
# --- Helper for timestamp formatting ---
def format_timestamp(seconds):
    ms = int((seconds - int(seconds)) * 1000)
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

# --- Write outputs ---
txt_lines = []
srt_lines = []

for i, seg in enumerate(segments, start=1):
    start_ts = format_timestamp(seg.start)
    end_ts = format_timestamp(seg.end)
    text = seg.text.strip()

    txt_lines.append(f"[{start_ts} --> {end_ts}] {text}")
    srt_lines.append(f"{i}\n{start_ts} --> {end_ts}\n{text}\n")
    print(f"[{start_ts} --> {end_ts}] {seg.text}")

In [ ]:
# # --- Save SRT file ---
with open(output_srt, "w", encoding="utf-8") as f:
    f.write("\n".join(srt_lines))

print(f"\n✅ Translation complete!")
print(f"SRT saved to: {output_srt}")

# Google translate

In [ ]:
import time
from deep_translator import GoogleTranslator
from opencc import OpenCC  # pip install opencc

# Converters
cc_s2t = OpenCC('s2t')  # Simplified → Traditional

def safe_translate(text, src="zh-CN", dest="en", delay=2):
    """
    Translate text from Chinese to English with indefinite retries until success.
    """
    if not text.strip():
        return ""
    
    # Limit text for API
    if len(text) > 4500:
        text = text[:4500]

    attempt = 0
    while True:
        attempt += 1
        try:
            translated = GoogleTranslator(source=src, target=dest).translate(text)
            return translated
        except Exception as e:
            print(f"⚠️ Translation failed (attempt {attempt}): {e}. Retrying in {delay}s...")
            time.sleep(delay)

def convert_and_translate(input_file, output_file):
    with open(input_file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    translated_lines = []

    for line in lines:
        stripped = line.strip()
        if not stripped:
            translated_lines.append("\n")
            continue

        # Check for timeframe [00:01:23 - 00:01:45]
        if stripped.startswith("[") and "]" in stripped:
            try:
                time_part, text_part = stripped.split("]", 1)
                text_part = text_part.strip()

                # 1️⃣ Simplified → Traditional
                text_traditional = cc_s2t.convert(text_part)

                # 2️⃣ Translate Traditional text to English
                text_english = safe_translate(text_traditional)

                # 3️⃣ Print progress
                print(f"{time_part}] {text_part} → {text_traditional} → {text_english}")

                # 4️⃣ Save English in output file
                translated_lines.append(f"{time_part}] {text_english}\n")
            except ValueError:
                print(f"⚠️ Skipping malformed line: {stripped}")
                translated_lines.append(stripped + "\n")
        else:
            # Lines without timeframes
            text_traditional = cc_s2t.convert(stripped)
            text_english = safe_translate(text_traditional)
            print(f"{stripped} → {text_traditional} → {text_english}")
            translated_lines.append(text_english + "\n")

    with open(output_file, "w", encoding="utf-8") as f:
        f.writelines(translated_lines)

    print(f"✅ Conversion & translation complete! Saved as {output_file}")


In [ ]:
input_file = input_file
output = output

In [ ]:
import pandas as pd 
import re

# -----------------------------
# 1. Read the subtitle files
# -----------------------------
with open(input_file, "r", encoding="utf-8") as f:
    chinese_lines = f.readlines()

with open(output, "r", encoding="utf-8") as f:
    english_lines = f.readlines()

# -----------------------------
# 2. Regex pattern to extract time and text
# -----------------------------
pattern = re.compile(r"\[(\d{2}:\d{2}:\d{2},\d{3}) --> (\d{2}:\d{2}:\d{2},\d{3})\] (.+)")

chinese_data = []
for line in chinese_lines:
    match = pattern.match(line.strip())
    if match:
        start, end, text = match.groups()
        chinese_data.append((start, end, text))

english_data = []
for line in english_lines:
    match = pattern.match(line.strip())
    if match:
        start, end, text = match.groups()
        english_data.append((start, end, text))

# -----------------------------
# 3. Combine into a DataFrame
# -----------------------------
data = []
for idx, ((c_start, c_end, c_text), (e_start, e_end, e_text)) in enumerate(zip(chinese_data, english_data), 1):
    # Optionally, you can check if timestamps match
    if c_start != e_start or c_end != e_end:
        print(f"Warning: timestamps do not match at index {idx}")
    data.append([idx, c_start, c_end, c_text, e_text])

df = pd.DataFrame(data, columns=["Index", "Start", "End", "Text", "Translated"])
df 

In [ ]:
df = df.reset_index(drop=True)
df.to_excel("Subtitle.xlsx", index=False)


# Subtitle to video

In [ ]:
import subprocess
from pathlib import Path
video_in = Path("video.mp4")
subs_srt = Path(output_srt)
video_out = Path("Subtitle.mp4")

# On Windows, escape backslashes or use raw strings.
cmd = [
    "ffmpeg",
    "-y",
    "-i", str(video_in),
    "-vf", f"subtitles={subs_srt.as_posix()}",
    "-c:a", "copy",        # keep original audio
    video_out.as_posix()
]

subprocess.run(cmd, check=True)